Labor Market Intervention Analysis in Argentina 

In [ ]:
import os
import pandas as pd 
import numpy as np
import json
import statsmodels.api as sm
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category = ConvergenceWarning)
%pip install pyfixest
import pyfixest as pf

In [10]:
path =  "/Users/amenbekele/Python/Argentina/clean"
analysis_full = pd.read_csv(os.path.join(path, "analysis_full.csv"))
analysis_survey = pd.read_csv(os.path.join(path, "analysis_survey.csv"))

/var/folders/7m/3gz__rk9335d32kdn5j9lmdc0000gn/T/ipykernel_39814/3555510690.py:2: DtypeWarning: Columns (42,75,76,78) have mixed types. Specify dtype option on import or set low_memory=False.
  analysis_full = pd.read_csv(os.path.join(path, "analysis_full.csv"))


In [11]:
print(analysis_full.shape)
print(analysis_full.columns)
print(analysis_full.head(10))
analysis_full.describe()

(103150, 133)
Index(['id_SSE', 'treat_phase', 'treat', 'provincia', 'agenciaterritorial',
       'municipio', 'treat_any', 'grupo', 'muni_id', 'acepta_gestion_oe',
       ...
       'n_applications_z', 'reservation_wage_z', 'days_worked_z',
       'job_prob_3m_z', 'emp_months_observed_post', 'emp_any_post',
       'emp_share_months_post', 'emp_any_pre', 'took_up', 'cv_exported'],
      dtype='object', length=133)
    id_SSE  treat_phase  treat     provincia agenciaterritorial  \
0   822751          1.0    0.0  buenos aires       bahia blanca   
1   303549          1.0    0.0  buenos aires       bahia blanca   
2   315796          1.0    0.0  buenos aires       bahia blanca   
3    47107          1.0    0.0  buenos aires       bahia blanca   
4   264668          1.0    0.0  buenos aires       bahia blanca   
5   397233          1.0    0.0  buenos aires       bahia blanca   
6   334073          1.0    0.0  buenos aires       bahia blanca   
7   681659          1.0    0.0  buenos aires   

,id_SSE,treat_phase,treat,treat_any,muni_id,edad,id_nivel_educativo,población2022indec,cantidadvataoctubre2024,cantidadfomentarft1yft2a,...,n_applications_z,reservation_wage_z,days_worked_z,job_prob_3m_z,emp_months_observed_post,emp_any_post,emp_share_months_post,emp_any_pre,took_up,cv_exported
count,1.031500e+05,103150.000000,103150.000000,103150.000000,103150.000000,103150.000000,103150.000000,103150.000000,103150.000000,103150.000000,...,985.000000,842.000000,1016.000000,1021.000000,6026.0,6026.000000,6026.000000,6026.000000,103150.000000,103150.000000
mean,5.058911e+05,1.808405,1.500068,0.750024,396.170422,32.925807,4.181367,428535.097218,19446.446728,16881.586534,...,0.003242,0.156566,0.205347,0.054563,3.0,0.566213,0.475053,0.870395,0.012399,0.002957
std,3.044735e+05,0.393558,1.118039,0.433001,127.897174,9.532741,1.304375,214781.142407,12357.141230,14236.829249,...,0.667432,1.533683,1.024537,0.986647,0.0,0.495638,0.457200,0.335896,0.110661,0.054297
min,1.100000e+01,1.000000,0.000000,0.000000,30.000000,18.000000,1.000000,6770.000000,6.000000,27.000000,...,-0.282612,-1.997403,-0.865278,-1.860946,3.0,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.400335e+05,2.000000,1.000000,1.000000,317.000000,25.000000,4.000000,268889.000000,4800.000000,3116.000000,...,-0.282612,-0.696019,-0.865278,-0.313382,3.0,0.000000,0.000000,1.000000,0.000000,0.000000
50%,5.022835e+05,2.000000,2.000000,1.000000,477.000000,31.000000,4.000000,488103.000000,27979.000000,4560.000000,...,-0.186086,-0.003793,0.077399,0.073509,3.0,1.000000,0.333333,1.000000,0.000000,0.000000
75%,7.658228e+05,2.000000,3.000000,1.000000,477.000000,40.000000,5.000000,598835.000000,27979.000000,31210.000000,...,0.006966,0.688433,1.020076,0.847292,3.0,1.000000,1.000000,1.000000,0.000000,0.000000
max,1.048625e+06,2.000000,3.000000,1.000000,589.000000,66.000000,7.000000,598835.000000,31646.000000,31210.000000,...,9.369967,15.225169,2.057021,1.621074,3.0,1.000000,1.000000,1.000000,1.000000,1.000000


In [12]:
#setting up the lasso search and test each with 5 fold cross validation
alpha_param = np.power(10, np.linspace(-4, 0, 30))
param_grid = {"lasso__alpha": alpha_param}
kfcv = KFold(n_splits = 5, random_state = 25, shuffle = True)

In [13]:
# Pick alpha by CV for one target, return the controls with nonzero coefficients
def lasso_keep(Xd, target):
    pipe = make_pipeline(StandardScaler(), Lasso(random_state = 1, max_iter = 5000))
    gs = GridSearchCV(pipe, param_grid, cv = kfcv, scoring = "neg_mean_squared_error")
    gs.fit(Xd, target)
    coefs = gs.best_estimator_.named_steps["lasso"].coef_
    return {c for c, b in zip(Xd.columns, coefs) if b != 0}

In [14]:
# Double selection: outcome plus each treatment, keep the union
def select_controls(d, y):
    dm = demean(d, [y] + treats + X)
    keep = set()
    for target in [y] + treats:
        keep |= lasso_keep(dm[X], dm[target])
    return sorted(keep)

In [15]:
# Load the variable lists from the json you saved
v = json.load(open(os.path.join(path, "analysis_vars.json")))
treats = v["treatments"]
X = v["x_candidates"]

In [16]:
# Subtract each stratum's average so the strata are accounted for before lasso
def demean(d, cols):
    return d[cols] - d.groupby("strata")[cols].transform("mean")

In [17]:
# How many blanks does each employment column have, and what values exist?
print(analysis_full[["emp_any_post", "emp_share_months_post"]].isna().sum())
print(analysis_full["emp_any_post"].value_counts(dropna = False))

emp_any_post             97124
emp_share_months_post    97124
dtype: int64
emp_any_post
NaN    97124
1.0     3412
0.0     2614
Name: count, dtype: int64


In [18]:
# People missing from the formal employment file were not formally employed
for c in ["emp_any_post", "emp_share_months_post"]:
    analysis_full[c] = analysis_full[c].fillna(0)
print(analysis_full["emp_any_post"].value_counts(dropna = False))

emp_any_post
0.0    99738
1.0     3412
Name: count, dtype: int64


In [19]:
# Test on one outcome and print the controls that were kept
y = "emp_any_post"
d = analysis_full.dropna(subset = [y] + X + ["strata"])
print("rows used:", len(d), "of", len(analysis_full))
sel = select_controls(d, y)
print(len(sel), "controls kept")
print(sel)

rows used: 103150 of 103150
20 controls kept
['any_etnia', 'any_pueblo_originario', 'discapacidad_SI', 'discapacidad_missing', 'edad', 'educ_secondary', 'experiencia_laboral_anterior_NO', 'experiencia_laboral_anterior_SI', 'female', 'has_children', 'hijos_menores_NO', 'hijos_menores_SI', 'nivel_educativo_Nunca_Asistió', 'nivel_educativo_Primario_Completo', 'nivel_educativo_Primario_Incompleto', 'nivel_educativo_Superior_Completo', 'nivel_educativo_Superior_Incompleto', 'trabaja_actualmente_SI', 'trabaja_actualmente_missing', 'work_experience']


In [20]:
# Make sure the fill is applied, then build the list of outcomes with their data
for c in ["emp_any_post", "emp_share_months_post"]:
    analysis_full[c] = analysis_full[c].fillna(0)

outcomes = []
for group, ys in v["admin_outcomes"].items():
    outcomes += [(group, y, analysis_full) for y in ys]
for group, ys in v["survey_outcomes"].items():
    outcomes += [(group, y, analysis_survey) for y in ys]

In [21]:
# Run lasso on one target, pick alpha by CV, return the kept controls
def lasso_keep(Xd, target):
    # Put the target on a common scale so lasso converges
    target = (target - target.mean()) / target.std()
    pipe = make_pipeline(StandardScaler(), Lasso(random_state = 1, max_iter = 5000))
    gs = GridSearchCV(pipe, param_grid, cv = kfcv, scoring = "neg_mean_squared_error")
    gs.fit(Xd, target)
    coefs = gs.best_estimator_.named_steps["lasso"].coef_
    return {c for c, b in zip(Xd.columns, coefs) if b != 0}


# Double selection: outcome plus each treatment, keep the union
def select_controls(d, y):
    dm = demean(d, [y] + treats + X)
    keep = set()
    for target in [y] + treats:
        keep |= lasso_keep(dm[X], dm[target])
    return sorted(keep)

In [22]:
# Run double selection for every outcome and save the kept controls
selected = {}
for group, y, data in outcomes:
    d = data.dropna(subset = [y] + X + ["strata"])
    print(f"{y} ({group}): rows used {len(d)} of {len(data)}")
    selected[y] = select_controls(d, y)
    print("   kept", len(selected[y]), "controls")

json.dump(selected, open(os.path.join(path, "selected_controls.json"), "w"), indent = 2)

enrolled_in_course (courses): rows used 103150 of 103150
   kept 19 controls
updated_cv (cv_update): rows used 103150 of 103150
   kept 21 controls
emp_any_post (formal_employment): rows used 103150 of 103150
   kept 20 controls
emp_share_months_post (formal_employment): rows used 103150 of 103150
   kept 17 controls
new_skill (skills_aspirations): rows used 1021 of 1021
   kept 8 controls
new_career_path (skills_aspirations): rows used 1021 of 1021
   kept 4 controls
portal_visits (job_search): rows used 967 of 1021
   kept 15 controls
search_hours (job_search): rows used 967 of 1021
   kept 17 controls
n_applications (job_search): rows used 985 of 1021
   kept 13 controls
reservation_wage (reservation_wage): rows used 842 of 1021
   kept 15 controls
days_worked (employment): rows used 1016 of 1021
   kept 4 controls
job_offer (employment): rows used 1021 of 1021
   kept 0 controls
job_prob_3m (employment): rows used 1021 of 1021
   kept 0 controls


In [ ]:
# Import pyfixest, define the helpers, fit one outcome, run Q3 and Q5
def clean_controls(sel):
    sel = list(sel)
    for base in ["hijos_menores", "experiencia_laboral_anterior", "trabaja_actualmente", "discapacidad"]:
        no, si = base + "_NO", base + "_SI"
        if no in sel and si in sel:
            sel.remove(no)
    return sel

def fit_ols(d, y, sel):
    d = d.dropna(subset = [y] + X + ["strata"]).copy()
    # pyfixest needs an explicit dtype for the fixed-effect column
    d["strata"] = d["strata"].astype("string")
    sel = clean_controls(sel)
    rhs = " + ".join(treats + sel)
    fit = pf.feols(f"{y} ~ {rhs} | strata", data = d, vcov = "hetero")
    return fit, d

def test_equal(fit, a, b):
    names = list(fit._coefnames)
    R = np.zeros((1, len(names)))
    R[0, names.index(a)] = 1
    R[0, names.index(b)] = -1
    return fit.wald_test(R = R)

fit, d = fit_ols(analysis_full, "emp_any_post", selected["emp_any_post"])
fit.summary()
print("Q3: T2 vs T1")
print(test_equal(fit, "SkillRecs", "CV"))
print("Q5: T3 vs T2")
print(test_equal(fit, "DemandRecs", "SkillRecs"))

###

Estimation:  OLS
Dep. var.: emp_any_post, Fixed effects: strata
sample: None = all
Inference:  hetero
Observations:  103150

| Coefficient                         |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:------------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| CV                                  |      0.001 |        0.002 |     0.439 |      0.660 | -0.002 |   0.004 |
| SkillRecs                           |      0.003 |        0.002 |     1.836 |      0.066 | -0.000 |   0.006 |
| DemandRecs                          |      0.002 |        0.002 |     1.528 |      0.126 | -0.001 |   0.005 |
| any_etnia                           |      0.007 |        0.002 |     2.702 |      0.007 |  0.002 |   0.012 |
| any_pueblo_originario               |      0.002 |        0.002 |     1.096 |      0.273 | -0.002 |   0.006 |
| discapacidad_SI                     |     -0.018 |        0.006 |    -3.127 |      0

/var/folders/7m/3gz__rk9335d32kdn5j9lmdc0000gn/T/ipykernel_39814/2863669635.py:18: UserWarning: 
            2 variables dropped due to multicollinearity.
            The following variables are dropped: ['hijos_menores_SI', 'work_experience'].
            
  fit = pf.feols(f"{y} ~ {rhs} | strata", data = d, vcov = "hetero")
/var/folders/7m/3gz__rk9335d32kdn5j9lmdc0000gn/T/ipykernel_39814/2863669635.py:26: UserWarning: Distribution changed to chi2, as R is not an identity matrix and q is not a zero vector.
  return fit.wald_test(R = R)
